<a href="https://colab.research.google.com/github/AmirJlr/Thesis/blob/master/examples/sider.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
# !pip install torch_geometric
# !pip install deepchem
# !pip install rdkit
# !pip install torchinfo
# !pip install molfeat

In [2]:
# !git clone https://github.com/AmirJlr/FDGNN.git

In [3]:
!python -m ipykernel install --user --name kernel3

Installed kernelspec kernel3 in C:\Users\TEMP.SAD.007\AppData\Roaming\jupyter\kernels\kernel3


In [4]:
import os
os.chdir('../')

In [5]:
!pwd

'pwd' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
!ls

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
import random
import numpy as np
import torch

SEED = 11
def seed_set(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_set(SEED)

In [8]:
# %load modules/data_handler.py
import numpy as np
import pandas as pd

import torch
from torch_geometric.data import Dataset, InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import from_smiles
from torch_geometric.utils import degree

import os
from tqdm.notebook import tqdm

import deepchem as dc

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.model_selection import train_test_split

from molfeat.calc import FPCalculator, RDKitDescriptors2D, Pharmacophore2D, Pharmacophore3D, RDKitDescriptors3D
import datamol as dm
from molfeat.trans import MoleculeTransformer

from sklearn.decomposition import PCA

import signal

from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict



def generate_graph_list(df, smiles_column, target_column):
    graph_list = []

    for i, smile in tqdm(enumerate(df[smiles_column])):
        g = from_smiles(smile)
        g.x = g.x.float()
        y = torch.tensor(df[target_column][i], dtype=torch.float).view(1, -1)
        g.y = y
        graph_list.append(g)

    return graph_list



############################# General Loader : #############################

def load_and_process_data(dataset, splitter="random", test_size=0.1, batch_size=32):
    """
    Loads a dataset, splits it into train, validation, and test sets, and creates PyTorch Geometric data loaders.
    """
    if splitter == "random":
        
        data_size = len(dataset)
        train_idx, test_idx = train_test_split(list(range(data_size)), test_size=0.1)
        train_idx, valid_idx = train_test_split(train_idx, test_size = test_size)  # Split train further into train and valid

        # Create data loaders for train, validation, and test sets
        train_loader = DataLoader(dataset[train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(dataset[valid_idx], batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(dataset[test_idx], batch_size=batch_size, shuffle=False)

    else:
        raise ValueError(f"Invalid splitter type: {splitter}. Valid options are 'random' or 'scaffold'.")

    return train_loader, val_loader, test_loader



def generate_scaffold(smiles, include_chirality=False):
    """Generate the Bemis-Murcko scaffold for a given SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol, includeChirality=include_chirality)
    return scaffold


def scaffold_split_indices(smiles_list, frac_train=0.8, frac_valid=0.1, frac_test=0.1, seed=None, include_chirality=False):
    """
    Perform scaffold splitting on a list of SMILES strings and return the indices for train, validation, and test sets.

    Args:
        smiles_list (list): List of SMILES strings.
        frac_train (float): Fraction of the dataset to use for training.
        frac_valid (float): Fraction of the dataset to use for validation.
        frac_test (float): Fraction of the dataset to use for testing.
        seed (int): Random seed for shuffling the scaffolds.
        include_chirality (bool): Whether to include chirality in scaffold generation.

    Returns:
        dict: Dictionary with train, valid, and test indices as torch tensors.
    """
    np.testing.assert_almost_equal(frac_train + frac_valid + frac_test, 1.0, err_msg="The fractions must sum to 1.")
    
    # Set random seed for reproducibility
    rng = np.random.RandomState(seed)
    
    # Group SMILES by their scaffold
    scaffolds = defaultdict(list)
    for ind, smiles in enumerate(smiles_list):
        scaffold = generate_scaffold(smiles, include_chirality)
        scaffolds[scaffold].append(ind)
    
    # Get scaffold keys and shuffle them
    scaffold_keys = list(scaffolds.keys())
    rng.shuffle(scaffold_keys)
    
    # Compute the number of samples for each set
    n_total = len(smiles_list)
    n_total_valid = int(np.floor(frac_valid * n_total))
    n_total_test = int(np.floor(frac_test * n_total))
    
    train_index = []
    valid_index = []
    test_index = []
    
    # Distribute the scaffold sets into train, valid, and test sets
    for scaffold_key in scaffold_keys:
        scaffold_set = scaffolds[scaffold_key]
        if len(valid_index) + len(scaffold_set) <= n_total_valid:
            valid_index.extend(scaffold_set)
        elif len(test_index) + len(scaffold_set) <= n_total_test:
            test_index.extend(scaffold_set)
        else:
            train_index.extend(scaffold_set)
    
    # Return indices as torch tensors in a dictionary
    return {
        'train': torch.tensor(train_index, dtype=torch.long),
        'valid': torch.tensor(valid_index, dtype=torch.long),
        'test': torch.tensor(test_index, dtype=torch.long)
    }
    
    
class FingerprintsDescriptorsCalculator:
    def __init__(self, smiles_column):
        self.smiles_column = smiles_column
        
        self.valid_molecules = []
        self.valid_smiles = []
        self.invalid_indices = []

        for index, smiles in tqdm(enumerate(self.smiles_column)) :
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print('******* Invalid Mol !!!!!!!')
                self.invalid_indices.append(index)
            else :
                self.valid_smiles.append(smiles)


        self.calc_ecfp = FPCalculator("ecfp")
        self.calc_topological = FPCalculator("topological")
        self.calc_maccs = FPCalculator("maccs")
        self.calc_estate = FPCalculator("estate")
        self.calc_rdkit2D = RDKitDescriptors2D(replace_nan=True)
        self.calc_phar2D = Pharmacophore2D()
      

        self.featurizer_ecfp = MoleculeTransformer(self.calc_ecfp, dtype=np.float64)
        self.featurizer_topological = MoleculeTransformer(self.calc_topological, dtype=np.float64)
        self.featurizer_maccs = MoleculeTransformer(self.calc_maccs, dtype=np.float64)
        self.featurizer_estate = MoleculeTransformer(self.calc_estate, dtype=np.float64)
        self.featurizer_rdkit2D = MoleculeTransformer(self.calc_rdkit2D, dtype=np.float64)
        self.featurizer_phar2D = MoleculeTransformer(self.calc_phar2D, dtype=np.float64)
        

    def calculate_ecfp(self):
        with dm.without_rdkit_log():
            return self.featurizer_ecfp(self.valid_smiles)

    def calculate_topological(self):
        with dm.without_rdkit_log():
            return self.featurizer_topological(self.valid_smiles)

    def calculate_maccs(self):
        with dm.without_rdkit_log():
            return self.featurizer_maccs(self.valid_smiles)

    def calculate_estate(self):
        with dm.without_rdkit_log():
            return self.featurizer_estate(self.valid_smiles)

    def calculate_rdkit2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_rdkit2D(self.valid_smiles)

    def calculate_phar2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_phar2D(self.valid_smiles)
    
    def get_invalid_indices(self):
        return self.invalid_indices

    def get_valid_smiles(self):
        return self.valid_smiles


# Usage Example :
# df = pd.read_csv('/content/bace.csv')
# smiles_column = df['mol'].values

# calculator = FingerprintsDescriptorsCalculator(smiles_column)

# ecfp = calculator.calculate_ecfp()
# topological = calculator.calculate_topological()
# maccs = calculator.calculate_maccs()
# estate = calculator.calculate_estate()
# rdkit2D = calculator.calculate_rdkit2D()
# phar2D = calculator.calculate_phar2D()

# phar3D = calculator.calculate_phar3D()
# rdkit3D = calculator.calculate_rdkit3D()
# invalid_indices = calculator.get_invalid_indices()


class FingerprintsDescriptorsCalculator2:
    def __init__(self, smiles_column):
        self.smiles_column = smiles_column
        
        self.valid_smiles = []
        self.invalid_indices = []

        for index, smiles in tqdm(enumerate(self.smiles_column)):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print(f'******* Invalid Mol at index {index} !!!!!!')
                self.invalid_indices.append(index)
            else:
                self.valid_smiles.append(smiles)

        self.calc_ecfp = FPCalculator("ecfp")
        self.calc_topological = FPCalculator("topological")
        self.calc_maccs = FPCalculator("maccs")
        self.calc_estate = FPCalculator("estate")
        self.calc_rdkit2D = RDKitDescriptors2D(replace_nan=True)
        self.calc_phar2D = Pharmacophore2D(replace_nan=True)

        self.featurizer_ecfp = MoleculeTransformer(self.calc_ecfp, dtype=np.float64)
        self.featurizer_topological = MoleculeTransformer(self.calc_topological, dtype=np.float64)
        self.featurizer_maccs = MoleculeTransformer(self.calc_maccs, dtype=np.float64)
        self.featurizer_estate = MoleculeTransformer(self.calc_estate, dtype=np.float64)
        self.featurizer_rdkit2D = MoleculeTransformer(self.calc_rdkit2D, dtype=np.float64)
        # self.featurizer_phar2D = MoleculeTransformer(self.calc_phar2D, dtype=np.float64)

    def calculate_phar2D(self, timeout=20):
        def timeout_handler(signum, frame):
            raise TimeoutError("Phar2D calculation timed out")

        signal.signal(signal.SIGALRM, timeout_handler)

        results = []
        remaining_smiles = []
        for index, smiles in tqdm(enumerate(self.valid_smiles)):
            signal.alarm(timeout)
            try:
                with dm.without_rdkit_log():
                    result = self.calc_phar2D(smiles)
                results.append(result)
                remaining_smiles.append(smiles)
            except TimeoutError:
                print(f"Phar2D calculation timed out for index {index}, smiles: {smiles}")
                self.invalid_indices.append(index)
            finally:
                signal.alarm(0)

        self.valid_smiles = remaining_smiles
        return np.array(results, dtype=np.float64)

    def calculate_ecfp(self):
        with dm.without_rdkit_log():
            return self.featurizer_ecfp(self.valid_smiles)

    def calculate_topological(self):
        with dm.without_rdkit_log():
            return self.featurizer_topological(self.valid_smiles)

    def calculate_maccs(self):
        with dm.without_rdkit_log():
            return self.featurizer_maccs(self.valid_smiles)

    def calculate_estate(self):
        with dm.without_rdkit_log():
            return self.featurizer_estate(self.valid_smiles)

    def calculate_rdkit2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_rdkit2D(self.valid_smiles)

    def get_invalid_indices(self):
        return self.invalid_indices

    def get_valid_smiles(self):
        return self.valid_smiles


# calculator = FingerprintsDescriptorsCalculator2(smiles_column)

# phar2D = calculator.calculate_phar2D()
# ecfp = calculator.calculate_ecfp()
# topological = calculator.calculate_topological()
# maccs = calculator.calculate_maccs()
# estate = calculator.calculate_estate()
# rdkit2D = calculator.calculate_rdkit2D()
# invalid_indices = calculator.get_invalid_indices()



class PCAReducer:
    def __init__(self, n_components=64):
        self.n_components = n_components
        self.pca_ecfp = PCA(n_components=self.n_components)
        self.pca_topological = PCA(n_components=self.n_components)
        self.pca_maccs = PCA(n_components=self.n_components)
        self.pca_estate = PCA(n_components=self.n_components)
        self.pca_rdkit2D = PCA(n_components=self.n_components)
        self.pca_phar2D = PCA(n_components=self.n_components)
        # self.pca_phar3D = PCA(n_components=self.n_components)
        # self.pca_rdkit3D = PCA(n_components=self.n_components)


    def reduce_ecfp(self, ecfp_data):
        return self.pca_ecfp.fit_transform(ecfp_data)

    def reduce_topological(self, topological_data):
        return self.pca_topological.fit_transform(topological_data)

    def reduce_maccs(self, maccs_data):
        return self.pca_maccs.fit_transform(maccs_data)

    def reduce_estate(self, estate_data):
        return self.pca_estate.fit_transform(estate_data)

    def reduce_rdkit2D(self, rdkit2D_data):
        return self.pca_rdkit2D.fit_transform(rdkit2D_data)

    def reduce_phar2D(self, phar2D_data):
        return self.pca_phar2D.fit_transform(phar2D_data)

    def reduce_phar3D(self, phar3D_data):
        return self.pca_phar3D.fit_transform(phar3D_data)

    def reduce_rdkit3D(self, rdkit3D_data):
        return self.pca_rdkit3D.fit_transform(rdkit3D_data)

# Usage Example :
# N_COMPONENTS = 64
# reducer = PCAReducer(n_components=N_COMPONENTS)

# ecfp_reduced = reducer.reduce_ecfp(ecfp)
# topological_reduced = reducer.reduce_topological(topological)
# maccs_reduced = reducer.reduce_maccs(maccs)
# estate_reduced = reducer.reduce_estate(estate)
# rdkit2D_reduced = reducer.reduce_rdkit2D(rdkit2D)
# phar2D_reduced = reducer.reduce_phar2D(phar2D)

# phar3D_reduced = reducer.reduce_phar3D(phar3D)
# rdkit3D_reduced = reducer.reduce_rdkit3D(rdkit3D)


class DTsetBasic(InMemoryDataset):
    def __init__(self, root, filename, smiles_column, label_column,
                 ECFP, Topological, MACCS, EState, Rdkit2D, Phar2D):
        self.filename = filename
        self.smiles_column = smiles_column
        # Allow label_column to be string or list of one string
        self.label_column = [label_column] if isinstance(label_column, str) else label_column

        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D

        super().__init__(root)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return [self.filename]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass

    def process(self):
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        graph_list = []
        for i, smiles in tqdm(enumerate(df[self.smiles_column]), desc="Processing SMILES"):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue

            g = from_smiles(smiles)
            g.x = g.x.float()

            # Extract label(s) — now always list
            label_vals = df.loc[i, self.label_column].values.astype(np.float32)
            g.y = torch.tensor(label_vals, dtype=torch.float).view(1, -1)  # Shape: [1, num_tasks=1]

            # Optional: Warn if NaN
            if torch.isnan(g.y).any():
                print(f"⚠️  NaN label at index {i} for SMILES: {smiles}")

            g.ECFP = torch.tensor(self.ECFP[i], dtype=torch.float).view(1, -1)
            g.Topological = torch.tensor(self.Topological[i], dtype=torch.float).view(1, -1)
            g.MACCS = torch.tensor(self.MACCS[i], dtype=torch.float).view(1, -1)
            g.EState = torch.tensor(self.EState[i], dtype=torch.float).view(1, -1)
            g.Rdkit2D = torch.tensor(self.Rdkit2D[i], dtype=torch.float).view(1, -1)
            g.Phar2D = torch.tensor(self.Phar2D[i], dtype=torch.float).view(1, -1)

            graph_list.append(g)

        data_list = graph_list

        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]
        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        self.save(data_list, self.processed_paths[0])

# dataset_64 = DTsetBasic(root='basic-64', filename='bace.csv', smiles_column='mol', label_column='Class',
#     ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
#     EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)



class DTsetBasicMulti(InMemoryDataset):
    def __init__(self, root, filename, smiles_column, label_columns,
                 ECFP, Topological, MACCS, EState, Rdkit2D, Phar2D):
        self.filename = filename
        self.smiles_column = smiles_column

        # اطمینان از اینکه label_columns حتماً یک لیست است
        self.label_columns = label_columns if isinstance(label_columns, list) else [label_columns]

        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D

        super().__init__(root)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return [self.filename]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass

    def process(self):
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        # Get all label columns: everything except smiles_column
        label_columns = [col for col in df.columns if col != self.smiles_column]

        graph_list = []
        for i, smiles in tqdm(enumerate(df[self.smiles_column]), desc="Processing SMILES"):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue

            g = from_smiles(smiles)
            g.x = g.x.float()

            # Extract all task labels
            # label_vals = df.loc[i, label_columns].values.astype(np.float32)

            # تغییر 2: استفاده از self.label_columns به جای استخراج اتوماتیک
            label_vals = df.loc[i, self.label_columns].values.astype(np.float32)
            g.y = torch.tensor(label_vals, dtype=torch.float).view(1, -1)  # Shape: [1, num_tasks]

            # Optional: Log if all labels missing
            if torch.isnan(g.y).all():
                print(f"⚠️  All labels NaN at index {i} for SMILES: {smiles}")

            g.ECFP = torch.tensor(self.ECFP[i], dtype=torch.float).view(1, -1)
            g.Topological = torch.tensor(self.Topological[i], dtype=torch.float).view(1, -1)
            g.MACCS = torch.tensor(self.MACCS[i], dtype=torch.float).view(1, -1)
            g.EState = torch.tensor(self.EState[i], dtype=torch.float).view(1, -1)
            g.Rdkit2D = torch.tensor(self.Rdkit2D[i], dtype=torch.float).view(1, -1)
            g.Phar2D = torch.tensor(self.Phar2D[i], dtype=torch.float).view(1, -1)

            graph_list.append(g)

        data_list = graph_list

        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]
        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        self.save(data_list, self.processed_paths[0])

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (d:\ProgramData\anaconda3\envs\pthgpu\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing 

In [9]:
from modules.data_handler import scaffold_split_indices, FingerprintsDescriptorsCalculator, PCAReducer, DTsetBasicMulti

In [10]:
import pandas as pd

df = pd.read_csv('data/datasets/sider.csv')
smiles_column = df['smiles'].values

In [11]:
calculator = FingerprintsDescriptorsCalculator(smiles_column)

phar2D = calculator.calculate_phar2D()
ecfp = calculator.calculate_ecfp()
topological = calculator.calculate_topological()
maccs = calculator.calculate_maccs()
estate = calculator.calculate_estate()
rdkit2D = calculator.calculate_rdkit2D()

invalid_indices = calculator.get_invalid_indices()
valid_smiles = calculator.get_valid_smiles()

0it [00:00, ?it/s]

[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
[20:00:38] WARNING: not removing hydrogen atom without neighbors
d:\ProgramData\anaconda3\envs\pthgpu\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finit

In [12]:
len(invalid_indices)

0

In [13]:
rdkit2D.shape

(1427, 223)

In [14]:
# Usage Example :
N_COMPONENTS = 64
reducer = PCAReducer(n_components=N_COMPONENTS)

ecfp_reduced = reducer.reduce_ecfp(ecfp)
topological_reduced = reducer.reduce_topological(topological)
maccs_reduced = reducer.reduce_maccs(maccs)
estate_reduced = reducer.reduce_estate(estate)
rdkit2D_reduced = reducer.reduce_rdkit2D(rdkit2D)
phar2D_reduced = reducer.reduce_phar2D(phar2D)

In [15]:
directory = 'data/sider/raw'
CSV_PATH = 'data/sider/raw/sider_cleaned.csv'

if not os.path.exists(directory):
    os.makedirs(directory)

df.drop(invalid_indices).to_csv(CSV_PATH, index=False)

In [16]:
label_columns = ['Hepatobiliary disorders', 'Metabolism and nutrition disorders', 'Product issues', 'Eye disorders', 'Investigations', 'Musculoskeletal and connective tissue disorders', 'Gastrointestinal disorders', 'Social circumstances', 'Immune system disorders', 'Reproductive system and breast disorders', 'Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'General disorders and administration site conditions', 'Endocrine disorders', 'Surgical and medical procedures', 'Vascular disorders', 'Blood and lymphatic system disorders', 'Skin and subcutaneous tissue disorders', 'Congenital, familial and genetic disorders', 'Infections and infestations', 'Respiratory, thoracic and mediastinal disorders', 'Psychiatric disorders', 'Renal and urinary disorders', 'Pregnancy, puerperium and perinatal conditions', 'Ear and labyrinth disorders', 'Cardiac disorders', 'Nervous system disorders', 'Injury, poisoning and procedural complications']

dataset = DTsetBasicMulti(root='data/sider', filename='sider_cleaned.csv', smiles_column='smiles',
    label_columns=label_columns,
    ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
    EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)

In [17]:
dataset[0]

Data(x=[13, 9], edge_index=[2, 24], edge_attr=[24, 3], smiles='C(CNCCNCCNCCN)N', y=[1, 27], ECFP=[1, 64], Topological=[1, 64], MACCS=[1, 64], EState=[1, 64], Rdkit2D=[1, 64], Phar2D=[1, 64])

In [18]:
from torch_geometric.loader import DataLoader

split_idx = scaffold_split_indices(valid_smiles, seed=SEED)
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=32, shuffle=False)
test_loader  = DataLoader(dataset[split_idx["test"]], batch_size=32, shuffle=False)

[20:13:04] WARNING: not removing hydrogen atom without neighbors
[20:13:04] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:05] WARNING: not removing hydrogen atom without neighbors
[20:13:06] WARNING: not removing hydrogen atom without neighbors
[20:13:06] WARNING: not removing hydrogen atom without neighbors


In [19]:
# %load modules/utils_classification.py
import os
import numpy as np
import torch
import torch.nn as nn
from torch import device
from torch.utils.data import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.nn import GINConv
from torch_geometric.nn import global_add_pool
from torch_geometric.loader import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim import Adam


from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from copy import deepcopy
from math import sqrt 
from tqdm.notebook import tqdm


def run_epoch_cls(model, optimizer, data_loader, loss_function, device, edge_attr, pass_data):
    """
    Runs a single training epoch for a PyG model on a graph property prediction task.

    Args:
        model (torch.nn.Module): The PyG model to be trained.
        optimizer (torch.optim.Optimizer, optional): The optimizer for training. Defaults to None.
        data_loader (torch_geometric.data.DataLoader): The data loader for the training data.
        loss_function (torch.nn.Module, optional): The loss function to use. Defaults to BCEWithLogitsLoss().
        device (str, optional): The device to use for training ("cpu" or "cuda"). Defaults to "cpu".

    Returns:
        tuple: A tuple containing the average loss and ROC-AUC score for the epoch.
    """

    model.to(device)
    model.train() if optimizer is not None else model.eval()

    y_true = []
    y_pred = []
    losses = []

    for step, data in enumerate(tqdm(data_loader, desc="Iteration")):  # Iterate in batches over the training dataset.
        data = data.to(device)  # Move data batch to device

        if edge_attr :
            if pass_data :
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch, data)
            else :
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        else :
            if pass_data :
                pred = model(data.x, data.edge_index, data.batch, data)
            else :
                pred = model(data.x, data.edge_index, data.batch)

        loss = loss_function(pred, data.y.to(torch.float32))  # Calculate loss

        if optimizer is not None:
            optimizer.zero_grad()  # Clear gradients
            loss.backward()  # Backpropagation
            optimizer.step()  # Update model parameters

        losses.append(loss.detach().cpu().numpy())
        y_true.append(data.y.view(pred.shape).detach().cpu())
        y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()

    # Calculate ROC-AUC score using sklearn
    auc_roc = roc_auc_score(y_true, y_pred)

    return np.array(losses).mean(), auc_roc




def train_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer):
    writer = SummaryWriter(f'runs/{tensorboard_writer}')

    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

    best_model = None
    best_val_auc = 0
    best_val_loss = float('inf')
    patience_counter = 0
    PATIENCE = 10  # Stop training if no improvement for 10 epochs

    for epoch in range(1, num_epochs + 1):
        train_loss, train_auc = run_epoch_cls(model, optimizer, train_loader, loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('auc/train', train_auc, epoch)

        val_loss, val_auc = run_epoch_cls(model, None, val_loader, loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('auc/val', val_auc, epoch)

        print(f'Epoch: {epoch:03d}, Train loss: {train_loss:.4f}, Train ROC-AUC: {train_auc:.4f}, Val loss: {val_loss:.4f}, Val ROC-AUC: {val_auc:.4f}')

        # Step the scheduler
        scheduler.step(val_loss)

        # Check for improvement
        if val_loss < best_val_loss:
            best_val_auc = val_auc
            best_val_loss = val_loss
            best_model = deepcopy(model)
            patience_counter = 0  # Reset counter
            print(f"✅ New best model saved at epoch {epoch} with Val Loss: {val_loss:.4f}")
        else:
            patience_counter += 1
            print(f"⚠️  No improvement. Patience: {patience_counter}/{PATIENCE}")

        # Early stopping check
        if patience_counter >= PATIENCE:
            print(f"🛑 Early stopping triggered at epoch {epoch}.")
            break

    writer.close()
    return {
        'best_model': best_model,
        'best_val_loss': best_val_loss,
        'best_val_auc': best_val_auc,
        'stopped_epoch': epoch  # Optional: return when training stopped
    }


# results = train_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer)
# best_model = results['best_model']
# best_val_rmse = results['best_val_rmse']

# # Save the best model
# torch.save(best_model.state_dict(), 'best_model.pth')

# # To load the model later
# # Instantiate the model class first (ensure the model class is defined the same way)
# model = YourModelClass()
# model.load_state_dict(torch.load('best_model.pth'))
# model.to(device)



######### Multi Task Classification #########

def multi_task_loss(pred, target, loss_function):
    """
    Compute multi-task loss ignoring NaN targets (missing labels).
    Assumes pred and target have shape [batch_size, num_tasks].
    """
    mask = ~torch.isnan(target)
    if mask.any():
        # Only compute loss where labels are present
        loss = loss_function(pred[mask], target[mask].to(torch.float32))
        return loss.mean()  # Reduce across all valid entries
    return torch.tensor(0.0, device=pred.device, requires_grad=True)


def run_epoch_multi_cls(model, optimizer, data_loader, loss_function, device, edge_attr, pass_data):
    """
    Runs a single epoch for multi-task classification.
    Handles missing labels (NaN) gracefully.
    Returns: average loss, average ROC-AUC across tasks (ignoring tasks with no valid labels).
    """
    model.to(device)
    model.train() if optimizer is not None else model.eval()

    y_true = []
    y_pred = []
    losses = []

    for step, data in enumerate(tqdm(data_loader, desc="Iteration")):
        data = data.to(device)

        # Forward pass
        if edge_attr:
            if pass_data:
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch, data)
            else:
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        else:
            if pass_data:
                pred = model(data.x, data.edge_index, data.batch, data)
            else:
                pred = model(data.x, data.edge_index, data.batch)

        # Compute loss
        loss = multi_task_loss(pred, data.y, loss_function)

        # Backward pass
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Collect for metrics
        losses.append(loss.detach().cpu().item())  # .item() for scalar
        y_true.append(data.y.detach().cpu())
        y_pred.append(pred.detach().cpu())

    # Concatenate all batches
    y_true = torch.cat(y_true, dim=0).numpy()  # Shape: [N, num_tasks]
    y_pred = torch.cat(y_pred, dim=0).numpy()  # Shape: [N, num_tasks]

    # Compute ROC-AUC per task
    auc_roc_list = []
    for i in range(y_true.shape[1]):
        mask = ~np.isnan(y_true[:, i])
        if mask.sum() > 1:  # Need at least one positive and one negative for AUC
            try:
                auc = roc_auc_score(y_true[mask, i], y_pred[mask, i])
                auc_roc_list.append(auc)
            except ValueError as e:
                print(f"⚠️  ROC AUC error for task {i}: {e}")
                auc_roc_list.append(np.nan)
        else:
            auc_roc_list.append(np.nan)

    # Average over valid tasks
    avg_auc_roc = np.nanmean(auc_roc_list) if len(auc_roc_list) > 0 else 0.0

    return np.mean(losses), avg_auc_roc


def train_multi_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer):
    """
    Train multi-task classification model with early stopping and LR scheduling.
    """
    writer = SummaryWriter(f'runs/{tensorboard_writer}')

    # Scheduler: Reduce LR when validation loss plateaus
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

    best_model = None
    best_val_auc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(1, num_epochs + 1):
        # Training
        train_loss, train_auc = run_epoch_multi_cls(
            model, optimizer, train_loader, loss_function, device, edge_attr, pass_data
        )
        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('auc/train', train_auc, epoch)

        # Validation
        val_loss, val_auc = run_epoch_multi_cls(
            model, None, val_loader, loss_function, device, edge_attr, pass_data
        )
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('auc/val', val_auc, epoch)

        print(f'Epoch {epoch:03d} | '
              f'Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f} | '
              f'Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}')

        # Step scheduler based on validation loss
        scheduler.step(val_loss)

        # Early stopping & model checkpointing
        if val_loss < best_val_loss:  
            best_val_auc = val_auc
            best_val_loss = val_loss
            best_model = deepcopy(model)
            patience_counter = 0
            print(f"✅ New best model (Val AUC: {val_auc:.4f}) at epoch {epoch}")
        else:
            patience_counter += 1
            print(f"⚠️  No improvement. Patience: {patience_counter}/{PATIENCE}")

        # if val_auc > best_val_auc:  
        #     best_val_auc = val_auc
        #     best_val_loss = val_loss 
        #     best_model = deepcopy(model)
        #     patience_counter = 0
        #     print(f"✅ New best model (Val AUC: {val_auc:.4f}) at epoch {epoch}")
        # else:
        #     patience_counter += 1
        #     print(f"⚠️ No improvement. Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"🛑 Early stopping at epoch {epoch}")
            break

    writer.close()

    return {
        'best_model': best_model,
        'best_val_loss': best_val_loss,
        'best_val_auc': best_val_auc,
        'stopped_epoch': epoch
    }

In [20]:
from modules.utils_classification import train_multi_cls, run_epoch_multi_cls

In [21]:
# %load models/GinGat.py
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import (
    GATConv, GINEConv, BatchNorm,
    global_mean_pool, global_max_pool, global_add_pool, GlobalAttention
)
from torch_geometric.data import Data, Batch


############### LSTM Pooling ###############
class LSTMAttentionPooling(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x, batch):
        num_graphs = batch.max().item() + 1
        pooled_outputs = []
        for i in range(num_graphs):
            node_embeds = x[batch == i].unsqueeze(0)
            h_0 = torch.zeros(self.lstm.num_layers, 1, self.lstm.hidden_size, device=x.device)
            c_0 = torch.zeros(self.lstm.num_layers, 1, self.lstm.hidden_size, device=x.device)
            lstm_out, _ = self.lstm(node_embeds, (h_0, c_0))
            attention_weights = F.softmax(self.attention(lstm_out.squeeze(0)), dim=0)
            graph_embedding = torch.sum(attention_weights * lstm_out.squeeze(0), dim=0)
            pooled_outputs.append(graph_embedding)
        return torch.stack(pooled_outputs, dim=0)


############### GRU Pooling ###############
class GRUAttentionPooling(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x, batch):
        pooled_outputs = []
        num_graphs = batch.max().item() + 1
        for i in range(num_graphs):
            nodes_in_graph = x[batch == i].unsqueeze(0)
            h_0 = torch.zeros(self.gru.num_layers, 1, self.gru.hidden_size, device=x.device)
            gru_out, _ = self.gru(nodes_in_graph, h_0)
            attention_weights = F.softmax(self.attention(gru_out.squeeze(0)), dim=0)
            graph_embedding = torch.sum(attention_weights * gru_out.squeeze(0), dim=0)
            pooled_outputs.append(graph_embedding)
        return torch.stack(pooled_outputs, dim=0)



class CGRUAttentionPooling(nn.Module):
    def __init__(self, input_dim, hidden_dim, processing_steps=3):
        super().__init__()
        self.processing_steps = processing_steps # T steps
        
        # استفاده از GRUCell به جای GRU
        # ورودی سلول: ویژگی استخراج شده از گراف (input_dim)
        # حالت پنهان سلول: همان بردار پرس‌وجو یا Query (hidden_dim)
        self.gru_cell = nn.GRUCell(input_dim, hidden_dim)
        
        # شبکه Attention: ترکیب ویژگی نودها و بردار Query برای محاسبه وزن
        self.attention = nn.Linear(input_dim + hidden_dim, 1)

    def forward(self, x, batch):
        pooled_outputs = []
        num_graphs = batch.max().item() + 1
        
        for i in range(num_graphs):
            # نودهای مربوط به یک گراف خاص
            nodes = x[batch == i]  # Shape: [num_nodes, input_dim]
            num_nodes = nodes.size(0)
            
            # مقداردهی اولیه بردار Query (q_0) با صفر
            q_t = torch.zeros(1, self.gru_cell.hidden_size, device=x.device)
            
            step_outputs = []
            
            # حلقه روی مراحل پردازش (T)، نه روی نودها!
            for t in range(self.processing_steps):
                # تکثیر بردار Query به تعداد نودها برای محاسبه Attention
                q_t_expanded = q_t.expand(num_nodes, -1) # Shape: [num_nodes, hidden_dim]
                
                # ترکیب ویژگی نودها با بردار Query مرحله فعلی
                attn_input = torch.cat([nodes, q_t_expanded], dim=-1)
                
                # محاسبه وزن‌های Attention برای تمام نودها به صورت همزمان
                attn_weights = F.softmax(self.attention(attn_input), dim=0) # [num_nodes, 1]
                
                # محاسبه o_t: جمع وزن‌دار نودها بر اساس Attention
                # این بخش کاملاً Permutation Invariant است
                o_t = torch.sum(attn_weights * nodes, dim=0, keepdim=True) # [1, input_dim]
                
                # به‌روزرسانی Query برای مرحله بعد توسط GRU
                q_t = self.gru_cell(o_t, q_t) # [1, hidden_dim]
                
                # ذخیره خروجی این مرحله
                step_outputs.append(o_t.squeeze(0))
            
            # اتصال خروجی تمام مراحل به هم (z_G = o_1 \oplus o_2 \dots \oplus o_T)
            graph_embedding = torch.cat(step_outputs, dim=-1) 
            pooled_outputs.append(graph_embedding)
            
        return torch.stack(pooled_outputs, dim=0)



############### Main Model (GINGAT) ###############
class GINGAT(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_channels, out_channels, heads,
                 dropout, pooling_type, num_tasks, use_dummy=True, feature_mode="both",
                 num_gin_layers=4, num_gat_layers=1):
        super().__init__()
        self.use_dummy = use_dummy
        self.pooling_type = pooling_type
        self.feature_mode = feature_mode
        self.num_gin_layers = num_gin_layers
        self.num_gat_layers = num_gat_layers

        self.out_channels = out_channels
        self.hidden_channels = hidden_channels

        # === Graph backbone ===
        self.graph_convs = nn.ModuleList()
        self.graph_bns = nn.ModuleList()

        for i in range(self.num_gin_layers):
            in_dim = node_dim if i == 0 else hidden_channels
            out_dim = hidden_channels if i < self.num_gin_layers - 1 else out_channels
            self.graph_convs.append(
                GINEConv(nn.Sequential(
                    nn.Linear(in_dim, out_dim), nn.ReLU(),
                    nn.Linear(out_dim, out_dim)
                ), edge_dim=edge_dim)
            )
            self.graph_bns.append(BatchNorm(out_dim))

        # === Graph Pooling Layer ===
        if pooling_type == 'lstm':
            self.pooling = LSTMAttentionPooling(out_channels, out_channels)
        elif pooling_type == 'gru':
            self.pooling = GRUAttentionPooling(out_channels, out_channels)
        elif pooling_type == 'attention':
            self.pooling = GlobalAttention(gate_nn=nn.Linear(out_channels, 1))
        elif pooling_type == 'mean':
            self.pooling = global_mean_pool
        elif pooling_type == 'max':
            self.pooling = global_max_pool
        elif pooling_type == 'sum':
            self.pooling = global_add_pool
        else:
            raise ValueError("Pooling must be one of 'lstm', 'gru', 'attention', 'mean', 'max', 'sum'")

        # === Dummy graph branch ===
        if self.use_dummy:
            self.node_convs = nn.ModuleList()
            self.node_bns = nn.ModuleList()

            for i in range(self.num_gat_layers):
                in_dim = out_channels if i == 0 else hidden_channels
                out_dim = hidden_channels
                self.node_convs.append(GATConv(in_dim, out_dim, heads=heads, concat=False))
                self.node_bns.append(BatchNorm(out_dim))

            if out_channels != hidden_channels:
                self.residual_proj = nn.Linear(out_channels, hidden_channels)
            else:
                self.residual_proj = None
        else:
            self.node_convs = None
            self.node_bns = None
            self.residual_proj = None
            self.ablation_proj = None

        # === Output head ===
        self.fc1 = nn.Linear(hidden_channels, hidden_channels // 2)
        self.fc2 = nn.Linear(hidden_channels // 2, num_tasks)
        self.dropout = nn.Dropout(dropout)

        self.last_attention = {}
        self.reset_parameters()

    def forward(self, x, edge_index, edge_attr, batch, data):
        device = x.device
        edge_attr = edge_attr.float().to(device)

        # === GNN Encoder ===
        for i, (conv, bn) in enumerate(zip(self.graph_convs, self.graph_bns)):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            if i < self.num_gin_layers - 1:
                x = self.dropout(x)

        graph_out = self.pooling(x, batch)

        # === Feature Selection ===
        if self.feature_mode == "fps":
            selected_features = [
                data.ECFP.to(device),
                data.Topological.to(device),
                data.MACCS.to(device),
                data.EState.to(device)
            ]
        elif self.feature_mode == "descs":
            selected_features = [
                data.Rdkit2D.to(device),
                data.Phar2D.to(device)
            ]
        elif self.feature_mode == "both":
            selected_features = [
                data.ECFP.to(device),
                data.Topological.to(device),
                data.MACCS.to(device),
                data.EState.to(device),
                data.Rdkit2D.to(device),
                data.Phar2D.to(device)
            ]
        else:
            raise ValueError(f"Invalid feature_mode: {self.feature_mode}.")

        features_2d = []
        for f in selected_features:
            if f.dim() == 1:
                features_2d.append(f.unsqueeze(1))
            else:
                features_2d.append(f.view(graph_out.size(0), -1))

        # === Apply Layer Normalization ===
        graph_out = F.layer_norm(graph_out, graph_out.size()[1:])
        normalized_features = [F.layer_norm(f, f.size()[1:]) for f in features_2d]

        if self.use_dummy:
            dummy_graphs = []
            for i in range(graph_out.size(0)):
                dummy_graph = self.create_complete_dummy_graph(
                    graph_out[i].unsqueeze(0),
                    [f[i].unsqueeze(0) for f in normalized_features],
                    device
                )
                dummy_graphs.append(dummy_graph)

            batched_dummy = Batch.from_data_list(dummy_graphs).to(device)
            x_dummy, edge_index_dummy = batched_dummy.x, batched_dummy.edge_index

            # === CRITICAL: Store the batch vector for attention visualization ===
            self.last_attention["batch"] = batched_dummy.batch

            # Initialize edge_index for the first GAT layer
            current_edge_index = edge_index_dummy

            # Apply GAT layers
            for i, (conv, bn) in enumerate(zip(self.node_convs, self.node_bns)):
                if i == 0 and self.residual_proj is not None:
                    initial_x = x_dummy

                # Pass the current edge_index to the GAT layer
                out = conv(x_dummy, current_edge_index, return_attention_weights=True)
                
                if isinstance(out, tuple):
                    x_dummy, (returned_edge_index, returned_alpha) = out
                    current_edge_index = returned_edge_index # Update for next layer
                    
                    # === CRITICAL FIX: Average across attention heads ===
                    if returned_alpha.dim() > 1:
                        returned_alpha = returned_alpha.mean(dim=1)  # Average over heads, keep per-edge dim
                    
                    # Only store from the LAST layer
                    if i == len(self.node_convs) - 1:
                        final_alpha = returned_alpha
                        final_edge_index = returned_edge_index
                else:
                    x_dummy = out
                    # If no attention returned, skip storing
                    if i == len(self.node_convs) - 1:
                        final_alpha = None
                        final_edge_index = None

                x_dummy = bn(x_dummy)
                x_dummy = F.relu(x_dummy)

                if i == 0 and self.residual_proj is not None:
                    x_dummy = x_dummy + self.residual_proj(initial_x)

            # Store attention from the FINAL GAT layer only
            self.last_attention["edge_index"] = final_edge_index.detach().cpu() if final_edge_index is not None else None
            self.last_attention["alpha"] = final_alpha.detach().cpu() if final_alpha is not None else None
        
            
            # Extract central node
            num_feats_per_graph = len(normalized_features)
            stride = num_feats_per_graph + 1
            central_indices = torch.arange(0, len(dummy_graphs) * stride, stride, device=device)
            x_processed = x_dummy[central_indices]

        else:
            feat_cat = torch.cat([graph_out] + normalized_features, dim=1)
            if self.ablation_proj is None:
                total_concat_dim = feat_cat.size(1)
                self.ablation_proj = nn.Linear(total_concat_dim, self.hidden_channels).to(device)
            x_processed = F.relu(self.ablation_proj(feat_cat))
            self.last_attention = None

        # === Final Prediction Head ===
        x_final = F.relu(self.fc1(x_processed))
        x_final = self.dropout(x_final)
        return self.fc2(x_final)

    def create_complete_dummy_graph(self, graph_embedding, features, device):
        node_features = torch.cat([graph_embedding] + features, dim=0)
        num_nodes = node_features.size(0)

        edge_list = []
        for i in range(num_nodes):
            for j in range(num_nodes):
                edge_list.append([i, j])

        edge_index = torch.tensor(edge_list, dtype=torch.long, device=device).t().contiguous()
        return Data(x=node_features, edge_index=edge_index)

    def reset_parameters(self):
        for conv, bn in zip(self.graph_convs, self.graph_bns):
            conv.reset_parameters()
            bn.reset_parameters()

        if hasattr(self.pooling, 'reset_parameters'):
            self.pooling.reset_parameters()
        elif self.pooling_type == 'lstm':
            self.pooling.lstm.reset_parameters()
            self.pooling.attention.reset_parameters()
        elif self.pooling_type == 'gru':
            self.pooling.gru.reset_parameters()
            self.pooling.attention.reset_parameters()

        if self.use_dummy:
            for conv, bn in zip(self.node_convs, self.node_bns):
                conv.reset_parameters()
                bn.reset_parameters()
            if self.residual_proj is not None:
                self.residual_proj.reset_parameters()
        else:
            if self.ablation_proj is not None:
                self.ablation_proj.reset_parameters()

        self.fc1.reset_parameters()
        self.fc2.reset_parameters()

In [22]:
from models.GinGat import GINGAT

In [23]:
import torch
from torchinfo import summary

EPOCHS = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOSS_FUNCTION = torch.nn.BCEWithLogitsLoss(reduction='none')  # Use reduction='none' to apply mask later


In [24]:
import optuna


def objective(trial):
    # Suggest hyperparameters
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 96, 128])
    heads = trial.suggest_categorical('heads', [2, 4, 6, 8])
    dropout = trial.suggest_float('dropout', 0.1, 0.6)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-2, log=True)
    gin_layers = trial.suggest_categorical('gin_layers', [3, 4, 5, 6])

    # Build model
    model = GINGAT(
        node_dim=9,
        edge_dim=3,
        hidden_channels=hidden_channels,
        out_channels=N_COMPONENTS,
        heads=heads, 
        dropout=dropout,
        pooling_type='gru',
        num_tasks=27,
        use_dummy=True,
        feature_mode='both',
        num_gin_layers=gin_layers,
        num_gat_layers=1
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Train
    results = train_multi_cls(
        model=model,
        optimizer=optimizer,
        loss_function=LOSS_FUNCTION,
        train_loader=train_loader,
        val_loader=valid_loader,
        num_epochs=EPOCHS,
        device=device,
        edge_attr=True,
        pass_data=True,
        tensorboard_writer=f"optuna_trial_{trial.number}"
    )

    best_model = results['best_model']

    # Evaluate on validation set
    _, val_auc = run_epoch_multi_cls(
        model=best_model, 
        optimizer=None, 
        data_loader=valid_loader,
        loss_function=LOSS_FUNCTION, 
        device=device, 
        edge_attr=True, 
        pass_data=True
    )

    # Clean up memory
    del model, optimizer, best_model
    torch.cuda.empty_cache()

    return val_auc

In [25]:
# Run optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("\n" + "="*50)
print("Best trial:")
print(f"Validation AUC: {study.best_trial.value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-19 20:13:07,025] A new study created in memory with name: no-name-e470e10a-ac92-401c-94db-8759cfbfd3bc
d:\ProgramData\anaconda3\envs\pthgpu\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Iteration:   0%|          | 0/36 [00:00<?, ?it/s]

Iteration:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 001 | Train Loss: 0.5325 | Train AUC: 0.5496 | Val Loss: 0.4926 | Val AUC: 0.5275
✅ New best model (Val AUC: 0.5275) at epoch 1


Iteration:   0%|          | 0/36 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 040 | Train Loss: 0.4738 | Train AUC: 0.6850 | Val Loss: 0.4835 | Val AUC: 0.5438
⚠️  No improvement. Patience: 10/10
🛑 Early stopping at epoch 40


Iteration:   0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-04-19 21:41:21,336] Trial 49 finished with value: 0.5589745816533329 and parameters: {'hidden_channels': 64, 'heads': 4, 'dropout': 0.12925798716116277, 'lr': 0.0001334294352298993, 'weight_decay': 0.00012443804411345054, 'gin_layers': 5}. Best is trial 45 with value: 0.6232314947367015.



Best trial:
Validation AUC: 0.6232
Best Hyperparameters:
hidden_channels: 96
heads: 4
dropout: 0.10499630462498857
lr: 0.0002657370191098141
weight_decay: 0.00014780108555648258
gin_layers: 6


In [26]:
# Train final model with best hyperparameters
print("\n" + "="*50)
print("Training final model with best hyperparameters...")

best_params = study.best_trial.params
final_model = GINGAT(
    node_dim=9,
    edge_dim=3,
    hidden_channels=best_params['hidden_channels'],
    out_channels=N_COMPONENTS,
    heads=best_params['heads'], 
    dropout=best_params['dropout'],
    pooling_type='gru',
    num_tasks=27,
    use_dummy=True,
    feature_mode='both',
    num_gin_layers=best_params['gin_layers'],
    num_gat_layers=1
).to(device)

final_optimizer = torch.optim.Adam(
    final_model.parameters(), 
    lr=best_params['lr'], 
    weight_decay=best_params['weight_decay']
)

final_results = train_multi_cls(
    model=final_model,
    optimizer=final_optimizer,
    loss_function=LOSS_FUNCTION,
    train_loader=train_loader,
    val_loader=valid_loader,
    num_epochs=EPOCHS,
    device=device,
    edge_attr=True,
    pass_data=True,
    tensorboard_writer="final_best_model"
)

# Test evaluation
best_final_model = final_results['best_model']
_, test_auc = run_epoch_multi_cls(
    model=best_final_model,
    optimizer=None,
    data_loader=test_loader,
    loss_function=LOSS_FUNCTION,
    device=device,
    edge_attr=True,
    pass_data=True
)

print(f"\nFinal Test AUC: {test_auc:.4f}")



Training final model with best hyperparameters...


d:\ProgramData\anaconda3\envs\pthgpu\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Iteration:   0%|          | 0/36 [00:00<?, ?it/s]

Iteration:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 001 | Train Loss: 0.6463 | Train AUC: 0.5191 | Val Loss: 0.5760 | Val AUC: 0.5053
✅ New best model (Val AUC: 0.5053) at epoch 1


...

Iteration:   0%|          | 0/36 [00:00<?, ?it/s]

Iteration:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 023 | Train Loss: 0.4446 | Train AUC: 0.7511 | Val Loss: 0.4830 | Val AUC: 0.5839
⚠️  No improvement. Patience: 10/10
🛑 Early stopping at epoch 23


Iteration:   0%|          | 0/5 [00:00<?, ?it/s]


Final Test AUC: 0.6166


In [27]:
# Save results
import pandas as pd
from datetime import datetime

results_df = pd.DataFrame([{
    'gin_layers': best_params['gin_layers'],
    'hidden_channels': best_params['hidden_channels'],
    'heads': best_params['heads'],
    'dropout': best_params['dropout'],
    'lr': best_params['lr'],
    'weight_decay': best_params['weight_decay'],
    'val_auc': study.best_trial.value,
    'test_auc': test_auc,
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}])

results_df.to_csv('added_datasets/sider_optuna_best_results.csv', index=False)
print("\nResults saved to sider_optuna_best_results.csv")


Results saved to sider_optuna_best_results.csv


In [28]:
# model_gru_dummy_both = GINGAT(node_dim=9,
#                               edge_dim=3,
#                               hidden_channels=96,
#                               out_channels=N_COMPONENTS,
#                               heads=4, dropout=0.5,
#                               pooling_type='gru',
#                               num_tasks=27,
#                               use_dummy=True,
#                               feature_mode='both',
#                               num_gin_layers=3,
#                               num_gat_layers=1)

# optimizer_gru_dummy_both = torch.optim.Adam(model_gru_dummy_both.parameters(), lr=0.001, weight_decay=0.0005)

# summary(model_gru_dummy_both)

In [29]:
# results_gru_dummy_both = train_multi_cls(model = model_gru_dummy_both,
#     optimizer = optimizer_gru_dummy_both,
#     loss_function = LOSS_FUNCTION,
#     train_loader = train_loader,
#     val_loader = valid_loader,
#     num_epochs = EPOCHS,
#     device = device,
#     edge_attr = True,
#     pass_data = True,
#     tensorboard_writer = "model_gru_dummy_both")

## Test Results

In [30]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

- ### results_gru_dummy_both

In [31]:
# best_gru_dummy_both = results_gru_dummy_both['best_model']

# _ , test_auc_gru_dummy_both = run_epoch_multi_cls(model = best_gru_dummy_both, optimizer=None, data_loader=test_loader,
#     loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

# print(f"Test Result :  AUC: {test_auc_gru_dummy_both:.4f}")